# Surrogate scaling laws & data ablations

This notebook answers: **what actually moves surrogate quality?**

We reuse the Phase-1 dataset from `manual_surrogate_run/dataset.h5` (no new OFT
solves). Experiments subsample that pool so you can sweep $N$, coverage, PCA
rank, and distribution shift cheaply.

**Primary score:** RMSE / mean-baseline (ratio). Also track R² and “within 5%”
accuracy-like %. See `manual_surrogate.ipynb` for metric definitions.

**Kernel:** select **Python (autotokamak)** / `venv/bin/python`.

## 0. Setup

In [ ]:
from __future__ import annotations

import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

HERE = Path.cwd().resolve()
REPO_ROOT = HERE if (HERE / "pyproject.toml").is_file() else HERE.parent
SRC = REPO_ROOT / "src"
if SRC.is_dir() and str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from autotokamak.surrogate.dataset import PARAM_ORDER, DatasetBundle, load_dataset
from autotokamak.surrogate.metrics import baseline_mean_predictor_rmse, summarize_psi_errors
from autotokamak.surrogate.reduce import fit_pca, inverse_transform, transform
from autotokamak.surrogate.zoo import make_model

DATA_H5 = REPO_ROOT / "notebooks" / "manual_surrogate_run" / "dataset.h5"
OUT_DIR = REPO_ROOT / "notebooks" / "scaling_runs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python → {sys.executable}")
print(f"Data   → {DATA_H5}  exists={DATA_H5.is_file()}")
print(f"Out    → {OUT_DIR}")

## 1. Load pool + freeze a held-out test set

Everything below trains on **subsets of the non-test pool** and scores on the
**same frozen test set**, so curves are comparable.

In [ ]:
assert DATA_H5.is_file(), f"Missing {DATA_H5}. Run manual_surrogate.ipynb first."

full = load_dataset(DATA_H5)
print(f"pool N={full.n_samples}, grid={full.grid_shape}, inputs={PARAM_ORDER}")

TEST_FRAC = 0.15
SEED = 0
rng = np.random.default_rng(SEED)
perm = rng.permutation(full.n_samples)
n_test = max(50, int(round(TEST_FRAC * full.n_samples)))
TEST_IDX = np.sort(perm[:n_test])
POOL_IDX = np.sort(perm[n_test:])
print(f"frozen test={TEST_IDX.size}, train-pool={POOL_IDX.size}")


def subset_bundle(bundle: DatasetBundle, idx: np.ndarray) -> DatasetBundle:
    return DatasetBundle(
        inputs=bundle.inputs[idx],
        psi=bundle.psi[idx],
        R=bundle.R,
        Z=bundle.Z,
        source_path=bundle.source_path,
    )


test_bundle = subset_bundle(full, TEST_IDX)


def fit_eval(
    train: DatasetBundle,
    test: DatasetBundle,
    *,
    model_name: str,
    hp: dict,
    n_pca: int,
) -> dict:
    # Train on train, score on test; return metric summary + timing.
    n_pca = int(min(n_pca, train.n_samples - 1, int(np.prod(train.grid_shape))))
    t0 = time.perf_counter()
    pca = fit_pca(train.psi, n_components=n_pca)
    y = transform(pca, train.psi)
    est = make_model(model_name, **hp)
    est.fit(train.inputs, y)
    pred = inverse_transform(pca, est.predict(test.inputs))
    baseline = baseline_mean_predictor_rmse(train.psi, test.psi)
    summary = summarize_psi_errors(test.psi, pred, baseline_rmse=baseline)
    summary.update(
        {
            "model": model_name,
            "n_train": int(train.n_samples),
            "n_pca": n_pca,
            "seconds": time.perf_counter() - t0,
            "hp": hp,
        }
    )
    return summary


DEFAULT_HP = {
    "poly_ridge": dict(alpha=1.0, degree=2),
    "kernel_ridge": dict(alpha=1e-2, gamma=1.0, kernel="rbf"),
    "mlp": dict(n_layers=1, layer_width=64, alpha=1e-4, learning_rate_init=1e-3),
}

## Experiment A — Learning curve (scaling with $N$)

Train on $N \in \{50, 100, 200, \ldots\}$ drawn from the pool; score on frozen test.
Expect: ratio ↓ and R² ↑ with $N$, then diminishing returns.

In [ ]:
# Shrink N_GRID for a quick smoke pass if needed
N_GRID = [50, 100, 200, 400, 800, 1600, 3200, 6400]
N_GRID = [n for n in N_GRID if n < POOL_IDX.size]
MODELS_A = ["poly_ridge", "kernel_ridge", "mlp"]
N_PCA_A = 12

rows_a = []
for n in N_GRID:
    train_idx = rng.choice(POOL_IDX, size=n, replace=False)
    train = subset_bundle(full, train_idx)
    for name in MODELS_A:
        s = fit_eval(train, test_bundle, model_name=name, hp=DEFAULT_HP[name], n_pca=N_PCA_A)
        rows_a.append(s)
        print(
            f"N={n:5d}  {name:14s}  ratio={s['rmse_vs_baseline']:.3f}  "
            f"R²={s['r2']:.3f}  ≤5%={s['pct_within_5pct']:.1f}%  t={s['seconds']:.1f}s"
        )

np.save(OUT_DIR / "expA_learning_curve.npy", np.array(rows_a, dtype=object), allow_pickle=True)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
metrics_plot = [
    ("rmse_vs_baseline", "RMSE / baseline (↓ better)", True),
    ("r2", "R² (↑ better)", False),
    ("pct_within_5pct", "within 5% of truth % (↑ better)", False),
]
for ax, (key, ylabel, logy) in zip(axes, metrics_plot):
    for name in MODELS_A:
        xs, ys = [], []
        for n in N_GRID:
            match = [r for r in rows_a if r["model"] == name and r["n_train"] == n]
            if match:
                xs.append(n)
                ys.append(match[0][key])
        ax.plot(xs, ys, marker="o", label=name)
    ax.set_xlabel("N_train")
    ax.set_ylabel(ylabel)
    ax.set_xscale("log", base=2)
    if logy:
        ax.set_yscale("log")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Exp A — scaling with dataset size", y=1.02)
fig.tight_layout()
plt.show()

## Experiment B — Input coverage / “evenness”

Same $N$, different ways to pick training points from the pool:

| Strategy | Idea |
|---|---|
| **random** | uniform subsample (control) |
| **space_filling** | greedy maximin — spread points across the 5D box |
| **clustered** | sample only from a few tight clusters (uneven coverage) |

Hypothesis: at fixed $N$, space-filling beats clustered; the gap shrinks as $N$ grows.

In [ ]:
def standardize_pool(inputs: np.ndarray) -> np.ndarray:
    mu = inputs.mean(axis=0)
    sig = inputs.std(axis=0)
    sig = np.where(sig < 1e-12, 1.0, sig)
    return (inputs - mu) / sig


def subsample_random(pool_idx: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    return rng.choice(pool_idx, size=n, replace=False)


def subsample_space_filling(pool_idx: np.ndarray, n: int, rng: np.random.Generator) -> np.ndarray:
    # Greedy maximin in standardized input space.
    X = standardize_pool(full.inputs[pool_idx])
    chosen_local = [int(rng.integers(0, len(pool_idx)))]
    dmin = np.linalg.norm(X - X[chosen_local[0]], axis=1)
    for _ in range(n - 1):
        j = int(np.argmax(dmin))
        chosen_local.append(j)
        dmin = np.minimum(dmin, np.linalg.norm(X - X[j], axis=1))
    return pool_idx[np.array(chosen_local)]


def subsample_clustered(
    pool_idx: np.ndarray, n: int, rng: np.random.Generator, n_clusters: int = 3
) -> np.ndarray:
    # Pick n points near a few random cluster centers (poor coverage).
    X = standardize_pool(full.inputs[pool_idx])
    centers = X[rng.choice(len(pool_idx), size=n_clusters, replace=False)]
    per = n // n_clusters
    rem = n - per * n_clusters
    chosen: list[int] = []
    for c_i, c in enumerate(centers):
        take = per + (1 if c_i < rem else 0)
        dist = np.linalg.norm(X - c, axis=1)
        local = np.argsort(dist)[:take]
        chosen.extend(local.tolist())
    return pool_idx[np.array(chosen)]


def coverage_score(idx: np.ndarray) -> dict:
    # Simple evenness diagnostics on standardized inputs.
    X = standardize_pool(full.inputs[idx])
    if len(idx) > 800:
        sub = X[rng.choice(len(idx), size=800, replace=False)]
    else:
        sub = X
    d = np.full(len(sub), np.inf)
    for i in range(len(sub)):
        dif = sub - sub[i]
        dif[i] = np.inf
        d[i] = np.min(np.linalg.norm(dif, axis=1))
    ents = []
    for j in range(X.shape[1]):
        hist, _ = np.histogram(X[:, j], bins=10, density=False)
        p = hist / max(hist.sum(), 1)
        p = p[p > 0]
        ents.append(float(-(p * np.log(p)).sum()))
    return {
        "mean_nn_dist": float(np.mean(d)),
        "mean_hist_entropy": float(np.mean(ents)),
    }

In [ ]:
N_B = [100, 400, 1600]
N_B = [n for n in N_B if n < POOL_IDX.size]
STRATEGIES = {
    "random": subsample_random,
    "space_filling": subsample_space_filling,
    "clustered": subsample_clustered,
}
MODEL_B = "kernel_ridge"
N_PCA_B = 12

rows_b = []
for n in N_B:
    for strat_name, fn in STRATEGIES.items():
        idx = fn(POOL_IDX, n, rng)
        cov = coverage_score(idx)
        train = subset_bundle(full, idx)
        s = fit_eval(
            train, test_bundle, model_name=MODEL_B, hp=DEFAULT_HP[MODEL_B], n_pca=N_PCA_B
        )
        s.update({"strategy": strat_name, **cov})
        rows_b.append(s)
        print(
            f"N={n:4d}  {strat_name:14s}  nn={cov['mean_nn_dist']:.3f}  "
            f"H={cov['mean_hist_entropy']:.3f}  ratio={s['rmse_vs_baseline']:.3f}  "
            f"R²={s['r2']:.3f}  ≤5%={s['pct_within_5pct']:.1f}%"
        )

np.save(OUT_DIR / "expB_coverage.npy", np.array(rows_b, dtype=object), allow_pickle=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for strat_name in STRATEGIES:
    xs = [r["n_train"] for r in rows_b if r["strategy"] == strat_name]
    ys = [r["rmse_vs_baseline"] for r in rows_b if r["strategy"] == strat_name]
    axes[0].plot(xs, ys, marker="o", label=strat_name)
    ys2 = [r["r2"] for r in rows_b if r["strategy"] == strat_name]
    axes[1].plot(xs, ys2, marker="o", label=strat_name)
axes[0].set_xlabel("N_train")
axes[0].set_ylabel("RMSE / baseline (↓)")
axes[0].set_title(f"Exp B — coverage ({MODEL_B})")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].set_xlabel("N_train")
axes[1].set_ylabel("R² (↑)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5.5, 4))
for strat_name, marker in zip(STRATEGIES, ["o", "s", "^"]):
    xs = [r["mean_nn_dist"] for r in rows_b if r["strategy"] == strat_name]
    ys = [r["rmse_vs_baseline"] for r in rows_b if r["strategy"] == strat_name]
    ax.scatter(xs, ys, marker=marker, s=60, label=strat_name)
ax.set_xlabel("mean NN distance in standardized input space (↑ = more spread)")
ax.set_ylabel("RMSE / baseline (↓)")
ax.set_title("Does better coverage → better model?")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## Experiment C — PCA rank ($n_{\mathrm{pca}}$)

Too few components → reconstruction floor. Too many → harder regression / overfit at small $N$.

In [ ]:
N_C = min(1600, POOL_IDX.size - 1)
PCA_GRID = [2, 4, 8, 12, 16, 24, 32]
train_c = subset_bundle(full, rng.choice(POOL_IDX, size=N_C, replace=False))
MODEL_C = "kernel_ridge"

rows_c = []
for n_pca in PCA_GRID:
    if n_pca >= train_c.n_samples:
        continue
    s = fit_eval(train_c, test_bundle, model_name=MODEL_C, hp=DEFAULT_HP[MODEL_C], n_pca=n_pca)
    rows_c.append(s)
    print(
        f"n_pca={n_pca:3d}  ratio={s['rmse_vs_baseline']:.3f}  "
        f"R²={s['r2']:.3f}  ≤5%={s['pct_within_5pct']:.1f}%"
    )

fig, ax = plt.subplots(figsize=(6, 3.8))
ax.plot([r["n_pca"] for r in rows_c], [r["rmse_vs_baseline"] for r in rows_c], marker="o")
ax.set_xlabel("n_pca")
ax.set_ylabel("RMSE / baseline (↓)")
ax.set_title(f"Exp C — PCA rank @ N={N_C} ({MODEL_C})")
ax.grid(True, alpha=0.3)
plt.show()

## Experiment D — Train/test distribution shift

Train only on the **low-$I_p$ half** of the pool; test on the **high-$I_p$ half**
(and the reverse). Measures whether the surrogate interpolates shaping but
fails outside the training current range — important for agent `regen_dataset` decisions.

In [ ]:
ip = full.inputs[:, PARAM_ORDER.index("Ip")]
ip_med = float(np.median(ip[POOL_IDX]))
low_pool = POOL_IDX[ip[POOL_IDX] <= ip_med]
high_pool = POOL_IDX[ip[POOL_IDX] > ip_med]
print(f"Ip median={ip_med:.0f} A  low_pool={low_pool.size}  high_pool={high_pool.size}")

N_D = min(800, low_pool.size, high_pool.size)
low_test_idx = rng.choice(low_pool, size=min(200, max(1, low_pool.size // 5)), replace=False)
high_test_idx = rng.choice(high_pool, size=min(200, max(1, high_pool.size // 5)), replace=False)
low_test = subset_bundle(full, low_test_idx)
high_test = subset_bundle(full, high_test_idx)

configs = [
    ("train_low→test_low", low_pool, low_test_idx, low_test),
    ("train_low→test_high", low_pool, low_test_idx, high_test),
    ("train_high→test_high", high_pool, high_test_idx, high_test),
    ("train_high→test_low", high_pool, high_test_idx, low_test),
]

rows_d = []
for label, pool, exclude, te in configs:
    avail = np.setdiff1d(pool, exclude)
    tr_idx = rng.choice(avail, size=min(N_D, avail.size), replace=False)
    tr = subset_bundle(full, tr_idx)
    s = fit_eval(tr, te, model_name="kernel_ridge", hp=DEFAULT_HP["kernel_ridge"], n_pca=12)
    s["shift"] = label
    rows_d.append(s)
    print(
        f"{label:22s}  ratio={s['rmse_vs_baseline']:.3f}  "
        f"R²={s['r2']:.3f}  ≤5%={s['pct_within_5pct']:.1f}%"
    )

fig, ax = plt.subplots(figsize=(8, 3.5))
labels = [r["shift"] for r in rows_d]
vals = [r["rmse_vs_baseline"] for r in rows_d]
ax.bar(labels, vals, color=["steelblue", "salmon", "steelblue", "salmon"])
ax.set_ylabel("RMSE / baseline (↓)")
ax.set_title("Exp D — Ip distribution shift")
ax.tick_params(axis="x", rotation=20)
ax.grid(True, axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

## Experiment E — Model ranking vs $N$ (who wins when?)

At small $N$, kernel methods often win; at large $N$, MLP may catch up (or not).
This plot is the practical guide for what the agent should try first.

In [ ]:
assert rows_a, "Run Exp A first"

fig, ax = plt.subplots(figsize=(7, 4))
for name in MODELS_A:
    xs = sorted({r["n_train"] for r in rows_a if r["model"] == name})
    ys = []
    for n in xs:
        ys.append(
            min(
                r["rmse_vs_baseline"]
                for r in rows_a
                if r["model"] == name and r["n_train"] == n
            )
        )
    ax.plot(xs, ys, marker="o", label=name)
ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xlabel("N_train")
ax.set_ylabel("RMSE / baseline (↓)")
ax.set_title("Exp E — which model wins at each N?")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

print(f"{'N':>6s}  winner          ratio")
for n in N_GRID:
    cand = [r for r in rows_a if r["n_train"] == n]
    if not cand:
        continue
    w = min(cand, key=lambda r: r["rmse_vs_baseline"])
    print(f"{n:6d}  {w['model']:14s}  {w['rmse_vs_baseline']:.3f}")

## Other experiments worth running (checklist)

Not automated here — add cells when needed:

1. **Isoflux / label quality filter** — train only on `isoflux_used=True` vs all successes.
2. **Grid resolution** — same $N$, coarser vs finer $(n_Z, n_R)$.
3. **Parameter-box width** — narrow shaping box vs full shipped box.
4. **Boosting models (P1)** — HistGB / ExtraTrees / PLS where KRR plateaus
   (`docs/surrogate_model_candidates.md`).
5. **Multi-seed variance** — repeat Exp A with seeds $\{0..4\}$; mean±std of ratio.
6. **Active learning** — add points where residual is largest vs random adds.
7. **Per-mode error** — RMSE of each PCA coefficient vs $N$.
8. **End-to-end timing** — wall-clock to reach ratio $< 0.3$ including OFT generation.

### How to read a “scaling law” here

On a log–log plot of RMSE-ratio vs $N$, a rough power law
$\mathrm{ratio} \sim N^{-\alpha}$ means $\alpha$ is your data exponent.
If $\alpha$ is flat, you need better features/models, not more of the same data.
If space-filling ≫ clustered at fixed $N$, invest in sampling design before model search.

## Summary — write a small JSON report

In [ ]:
def best_at(rows, n, key="rmse_vs_baseline"):
    cand = [r for r in rows if r.get("n_train") == n]
    if not cand:
        return None
    w = min(cand, key=lambda r: r[key])
    return {
        "model": w["model"],
        "ratio": w["rmse_vs_baseline"],
        "r2": w["r2"],
        "pct_within_5pct": w["pct_within_5pct"],
    }


report = {
    "data": str(DATA_H5),
    "n_pool": int(full.n_samples),
    "n_test": int(TEST_IDX.size),
    "expA_learning_curve": {
        "n_grid": N_GRID,
        "best_per_n": {str(n): best_at(rows_a, n) for n in N_GRID},
    },
    "expB_coverage": [
        {
            "strategy": r["strategy"],
            "n_train": r["n_train"],
            "ratio": r["rmse_vs_baseline"],
            "r2": r["r2"],
            "mean_nn_dist": r["mean_nn_dist"],
        }
        for r in rows_b
    ],
    "expC_pca": [
        {"n_pca": r["n_pca"], "ratio": r["rmse_vs_baseline"], "r2": r["r2"]} for r in rows_c
    ],
    "expD_shift": [
        {"shift": r["shift"], "ratio": r["rmse_vs_baseline"], "r2": r["r2"]} for r in rows_d
    ],
}
out = OUT_DIR / "scaling_report.json"
out.write_text(json.dumps(report, indent=2))
print(f"Wrote {out}")
print(json.dumps(report["expA_learning_curve"]["best_per_n"], indent=2))

## Results summary (this run)

Evidence below is from the completed run on `manual_surrogate_run/dataset.h5`
($N_{\mathrm{pool}}=9993$, frozen test $=1499$). Primary score: **RMSE / mean-baseline** (↓ better).

### Exp A — Scaling with $N$

**Question:** Does adding more training data improve the surrogate, and which model benefits?

**Conclusion:** Yes for **kernel_ridge**; weakly then not for the others.
- At small $N$ ($50$–$200$), **poly_ridge** wins (ratio $\approx 0.57 \to 0.49$).
- From $N \ge 400$, **kernel_ridge** takes over and keeps improving to ratio $\approx 0.27$ at $N=6400$ (R² $\approx 0.96$, within-5% $\approx 42\%$).
- **poly_ridge** plateaus near ratio $\approx 0.5$ / within-5% $\approx 20\%$ — more data does not help much.
- **mlp** stays worst (ratio near or above $1$ for most of the curve); it is not competitive in this PoC zoo.

### Exp B — Input coverage / evenness

**Question:** At fixed $N$, does a more even spread of inputs beat clustered sampling?

**Conclusion:** Yes — coverage matters a lot, especially at small–medium $N$.
- **space_filling** best at every $N$ tested (e.g. $N=100$: ratio $0.55$ vs random $0.70$ vs clustered $0.96$).
- **clustered** nearly fails to beat baseline even at $N=1600$ (ratio $\approx 0.81$).
- Higher mean nearest-neighbor distance (more spread) tracks lower ratio. Sampling design is a first-class lever, not just $N$.

### Exp C — PCA rank

**Question:** How many PCA components are enough?

**Conclusion:** Gains saturate around **12–16** components (at $N=1600$, kernel_ridge).
- $n_{\mathrm{pca}}=2 \to 12$: ratio $0.62 \to 0.31$.
- $16$–$32$: only tiny moves (ratio $\approx 0.29$–$0.30$). Extra components past $\sim 16$ are not worth the cost here.

### Exp D — $I_p$ distribution shift

**Question:** Does the model generalize across plasma-current ranges it never saw in training?

**Conclusion:** No — same-$I_p$-half works; cross-half fails.
- In-distribution (low→low / high→high): ratio $\approx 0.38$–$0.40$, R² $\approx 0.92$.
- Out-of-distribution (low→high / high→low): ratio $\approx 0.94$, R² collapses ($0.28$ / negative).
- The surrogate interpolates within the trained $I_p$ band; it does **not** extrapolate across current. Dataset regen / coverage must include the target $I_p$ range.

### Exp E — Who wins at each $N$?

**Question:** Which zoo model should the agent prefer as a function of dataset size?

**Conclusion:** **poly_ridge** for tiny data; **kernel_ridge** once $N \gtrsim 400$; never prefer this **mlp** on this evidence.
- Winner table: $N=50,100,200$ → poly_ridge; $N=400$–$6400$ → kernel_ridge.

### Bottom line for the project

1. Prefer **more + well-spread data** over swapping in the PoC MLP.
2. Default strong model: **kernel_ridge** (after a few hundred samples).
3. Keep **$n_{\mathrm{pca}} \approx 12$–$16$**.
4. Ensure training covers the **$I_p$ (and shaping) region** you will query — shift kills performance.